# 教程: 数据预处理 - 时间聚合

## 目的
在进行水文模拟时，我们经常会遇到原始数据的频率与模型所需的计算步长不匹配的情况。例如，我们可能有小时级别的降雨数据，但希望以天为步长来运行一个概念性水文模型。这就需要对原始数据进行**时间聚合（Temporal Aggregation）**。

本教程将展示一个常见的数据预处理任务：如何将小时降雨数据聚合为日总降雨量，以便用于日步长的水文模型。

此笔记本将：
1.  使用`pandas`库读取一个包含小时时间戳的CSV文件。
2.  演示如何使用`pandas.DataFrame.resample()`这一强大的功能，将数据从小时频率聚合到天频率。
3.  将聚合后的日数据作为输入，运行一个简单的水文模型。
4.  通过可视化，对比原始数据和聚合后数据的形态。

## 1. 环境设置

我们只需要`pandas`用于数据处理，`matplotlib`用于绘图，以及我们自己的一个简单水文模型用于演示。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting

## 2. 加载和解析小时数据

我们加载之前创建的`data/hourly_rainfall.csv`文件。关键步骤是：
- 使用`pd.read_csv`加载数据。
- 使用`pd.to_datetime`将`timestamp`列转换为pandas的`datetime`对象。
- 将`timestamp`列设为DataFrame的索引（`index`），这是进行时间序列分析的前提。

In [ ]:
hourly_df = pd.read_csv('../data/hourly_rainfall.csv')
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'])
hourly_df = hourly_df.set_index('timestamp')

print("加载的原始小时数据:")
display(hourly_df.head())

## 3. 进行时间聚合

现在我们使用`resample()`方法。我们提供`'D'`作为参数，表示我们希望将数据重新采样到**天（Daily）**的频率。然后，我们链式调用`.sum()`方法，表示我们希望将每个时间窗口（即每一天）内的所有数据点（即所有小时的降雨量）相加，得到日总降雨量。

In [ ]:
daily_df = hourly_df.resample('D').sum()

print("聚合后的日总降雨量:")
display(daily_df)

## 4. 可视化对比

为了更清楚地看到聚合的效果，我们把小时数据（柱状图）和聚合后的日数据（点线图）绘制在一起。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# 绘制小时数据
ax.bar(hourly_df.index, hourly_df['rainfall_mm'], width=0.04, color='c', alpha=0.7, label='Hourly Rainfall')

# 绘制日数据
ax.plot(daily_df.index, daily_df['rainfall_mm'], 'r-o', markersize=8, label='Aggregated Daily Rainfall')

ax.set_title('Temporal Aggregation: Hourly to Daily')
ax.set_xlabel('Date')
ax.set_ylabel('Rainfall (mm)')
ax.legend()
ax.grid(True, linestyle='--')
plt.show()

## 5. 使用聚合后的数据运行模型

最后，我们可以放心地将聚合后的日数据传递给一个日步长的水文模型。

In [ ]:
daily_rainfall = daily_df['rainfall_mm'].values

params = {'CN': 75, 'k_q': 0.8, 'k_s': 0.2}
runoff_module = SCSCurveNumberModule(CN=params['CN'])
routing_module = SimpleRouting(k_q=params['k_q'], k_s=params['k_s'])
model = HydrologicalModel(runoff_module, routing_module)

daily_outflows = [model.run(rain, pet=1.0) for rain in daily_rainfall]

print("使用日数据成功运行模型。")
print(f"日径流序列: {np.round(daily_outflows, 2)}")